<div dir="rtl" style="text-align:right">

# الفصل الثالث: Classification (التصنيف)
### من كتاب *Hands-On Machine Learning with Scikit-Learn & TensorFlow*

النوتبوك ده شرح بالعربي لمحتوى الفصل + الكود بتاعه شغّال على النسخ الحديثة من scikit-learn.

**إيه اللي هنتعلمه في الفصل؟**

| القسم | الفكرة الأساسية |
|---|---|
| MNIST | الداتا اللي هنشتغل عليها (صور أرقام مكتوبة بخط اليد) |
| Binary Classifier | تصنيف لفئتين بس: "5" أو "مش 5" |
| Performance Measures | إزاي نقيس جودة الـ classifier (Accuracy, Confusion Matrix, Precision, Recall, F1, ROC) |
| Multiclass | تصنيف لأكتر من فئتين (الأرقام من 0 لـ 9) |
| Error Analysis | نحلّل أخطاء الموديل عشان نحسّنه |
| Multilabel | كل صورة ممكن يبقى ليها أكتر من label |
| Multioutput | كل label ممكن يبقى له أكتر من قيمة (مثال: إزالة النويز من الصور) |

> **ملاحظة مهمة:** الكتاب اتكتب سنة 2017، فاضطريت أعدّل شوية حاجات في الكود عشان يشتغل اليوم
> (مثلًا `fetch_mldata` اتشالت من scikit-learn وبقت `fetch_openml`). كل تعديل هتلاقي جنبه تعليق بيشرحه.

</div>

<div dir="rtl" style="text-align:right">

## 0) التجهيزات (Setup)

بنستورد المكتبات الأساسية. وبنثبّت `np.random.seed(42)` عشان النتايج تبقى قابلة للتكرار.

</div>

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

np.random.seed(42)

mpl.rc("axes", labelsize=14)
mpl.rc("xtick", labelsize=12)
mpl.rc("ytick", labelsize=12)

<div dir="rtl" style="text-align:right">

## 1) MNIST

**MNIST** هي مجموعة من **70,000 صورة صغيرة** لأرقام مكتوبة بخط اليد (كتبها طلبة ثانوي وموظفين في مكتب الإحصاء الأمريكي).
كل صورة معاها label بيقول الرقم اللي فيها.

الداتا دي اتدرست كتير جدًا لدرجة إنها بتتسمّى **"Hello World" بتاع الـ Machine Learning**؛ أي حد يعمل algorithm تصنيف جديد أول حاجة بيجرّبه عليها MNIST.

الداتا اللي بتنزّلها scikit-learn بتبقى على شكل dictionary فيه:
- `data`: مصفوفة، كل صف = صورة، وكل عمود = feature (يعني بكسل)
- `target`: الـ labels (الرقم الحقيقي)
- `DESCR`: وصف الداتا

</div>

In [ ]:
from sklearn.datasets import fetch_openml

# تعديل: الكتاب بيستخدم fetch_mldata('MNIST original') وهي اتشالت.
# البديل الحديث هو fetch_openml (محتاجة إنترنت أول مرة بس).
mnist = fetch_openml("mnist_784", version=1, as_frame=False)

X, y = mnist["data"], mnist["target"]
y = y.astype(np.uint8)   # الـ labels بتيجي كنصوص "5" فبنحوّلها لأرقام

print("X shape:", X.shape)
print("y shape:", y.shape)

<div dir="rtl" style="text-align:right">

**يعني إيه `(70000, 784)`؟**
- 70,000 صورة.
- كل صورة **28×28 بكسل = 784 pixel**، وكل pixel هو feature واحد بيمثّل **شدة اللون** من 0 (أبيض) لـ 255 (أسود).

عشان نعرض صورة: ناخد الـ feature vector بتاعها، نعمله `reshape(28, 28)`، ونعرضه بـ `imshow()`.

</div>

In [ ]:
some_digit = X[36000]
some_digit_image = some_digit.reshape(28, 28)

plt.imshow(some_digit_image, cmap=mpl.cm.binary, interpolation="nearest")
plt.axis("off")
plt.show()

print("الـ label الحقيقي:", y[36000])

<div dir="rtl" style="text-align:right">

شكلها 5، والـ label بيأكد كده. نجرّب نعرض كذا رقم مع بعض عشان نحس بصعوبة المهمة (ناس بتكتب الأرقام بطرق مختلفة جدًا).

</div>

In [ ]:
def plot_digits(instances, images_per_row=10, **options):
    # دالة مساعدة: بترصّ مجموعة صور في شبكة واحدة
    size = 28
    images_per_row = min(len(instances), images_per_row)
    images = [instance.reshape(size, size) for instance in instances]
    n_rows = (len(instances) - 1) // images_per_row + 1
    row_images = []
    n_empty = n_rows * images_per_row - len(instances)
    images.append(np.zeros((size, size * n_empty)))
    for row in range(n_rows):
        rimages = images[row * images_per_row : (row + 1) * images_per_row]
        row_images.append(np.concatenate(rimages, axis=1))
    image = np.concatenate(row_images, axis=0)
    plt.imshow(image, cmap=mpl.cm.binary, **options)
    plt.axis("off")

def plot_digit(data):
    plt.imshow(data.reshape(28, 28), cmap=mpl.cm.binary, interpolation="nearest")
    plt.axis("off")

plt.figure(figsize=(9, 9))
example_images = X[:100]
plot_digits(example_images, images_per_row=10)
plt.show()

<div dir="rtl" style="text-align:right">

### Train/Test Split

قاعدة ذهبية: **اعمل test set واحفظه على جنب قبل ما تبص على الداتا بتاعتك بتفصيل.**

MNIST أصلًا متقسمة: أول **60,000** صورة للتدريب، وآخر **10,000** للاختبار.

### ليه بنعمل Shuffle للـ training set؟
1. عشان كل **cross-validation fold** يبقى متشابه (ما ينفعش fold يكون ناقصه أرقام معينة).
2. بعض الـ algorithms حسّاسة لترتيب الداتا وبتتعب لو شافت عيّنات متشابهة ورا بعض.

> ⚠️ الـ shuffle مش دايمًا فكرة كويسة، مثلًا في **Time Series** (أسعار الأسهم، الطقس) الترتيب نفسه مهم.

</div>

In [ ]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

shuffle_index = np.random.permutation(60000)
X_train, y_train = X_train[shuffle_index], y_train[shuffle_index]

<div dir="rtl" style="text-align:right">

## 2) تدريب Binary Classifier

هنبسّط المشكلة: بدل ما نعرف الـ 10 أرقام، هنعمل **"5-detector"** يفرّق بين فئتين بس: **5** و **مش 5**.
ده اسمه **Binary Classifier**.

بنجهّز الـ target vectors: `True` لو الرقم 5، و`False` لأي رقم تاني.

</div>

In [ ]:
y_train_5 = (y_train == 5)
y_test_5 = (y_test == 5)

print("نسبة الـ 5 في الداتا:", y_train_5.mean())

<div dir="rtl" style="text-align:right">

لاحظ إن حوالي **9%** بس من الصور هي 5. دي داتا **skewed** (فئة أكتر بكتير من التانية)، وده هيبقى مهم جدًا في الشرح اللي جاي.

### اختيار الـ Classifier: `SGDClassifier`
**SGD = Stochastic Gradient Descent.** مميزاته:
- بيتعامل مع الداتا الكبيرة بكفاءة.
- بيعالج كل عيّنة لوحدها (واحدة واحدة)، وده بيخليه مناسب كمان للـ **online learning**.
- اسمه *Stochastic* لأن فيه عشوائية في التدريب، فلازم تحدد `random_state` عشان النتايج تتكرر.

</div>

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)

sgd_clf.predict([some_digit])

<div dir="rtl" style="text-align:right">

الموديل خمّن إن الصورة دي **5** (`True`) وطلع صح. بس عيّنة واحدة مش دليل. لازم نقيس الأداء بطريقة سليمة.

</div>

<div dir="rtl" style="text-align:right">

## 3) Performance Measures (مقاييس الأداء)

تقييم الـ classifier **أصعب بكتير** من تقييم الـ regressor، وعشان كده الفصل بيقضي وقت طويل في الجزء ده. فيه مقاييس كتير هنتعلمها.

### 3.1) قياس الـ Accuracy باستخدام Cross-Validation

**K-fold cross-validation:** بنقسم الـ training set لـ K أجزاء (folds). بنخلّي كل جزء مرة test، وبندرّب على الباقي، ونقيس الأداء.

الأول هنعملها **بإيدنا** عشان نفهم اللي بيحصل جوّه `cross_val_score()` (ممكن تحتاج التحكم ده في حالات خاصة):
- `StratifiedKFold` بيعمل **stratified sampling**: كل fold فيه نفس نسبة الفئات تقريبًا.
- في كل لفّة: بنعمل **clone** من الموديل، ندرّبه على الـ training folds، ونتوقّع على الـ test fold، ونحسب نسبة التوقعات الصح.

</div>

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

# تعديل: النسخ الحديثة بتطلب shuffle=True لما تحدد random_state
skfolds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for train_index, test_index in skfolds.split(X_train, y_train_5):
    clone_clf = clone(sgd_clf)
    X_train_folds = X_train[train_index]
    y_train_folds = y_train_5[train_index]
    X_test_fold = X_train[test_index]
    y_test_fold = y_train_5[test_index]

    clone_clf.fit(X_train_folds, y_train_folds)
    y_pred = clone_clf.predict(X_test_fold)
    n_correct = sum(y_pred == y_test_fold)
    print(n_correct / len(y_pred))

<div dir="rtl" style="text-align:right">

والنسخة المختصرة اللي بتعمل نفس الحاجة: `cross_val_score()`.

</div>

In [ ]:
from sklearn.model_selection import cross_val_score

cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy")

<div dir="rtl" style="text-align:right">

**واو! أكتر من 95% accuracy في كل الـ folds؟** شكلها ممتازة... بس استنى.

خلينا نجرّب classifier **غبي جدًا**: بيقول على أي صورة إنها "مش 5" على طول، من غير ما يبص عليها أصلًا.

</div>

In [ ]:
from sklearn.base import BaseEstimator

class Never5Classifier(BaseEstimator):
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return np.zeros((len(X),), dtype=bool)

never_5_clf = Never5Classifier()
cross_val_score(never_5_clf, X_train, y_train_5, cv=3, scoring="accuracy")

<div dir="rtl" style="text-align:right">

### 💡 الدرس الأول (مهم جدًا)

الـ classifier الغبي ده جاب **أكتر من 90% accuracy!** السبب بسيط: حوالي 10% بس من الصور 5، فلو قلت دايمًا "مش 5" هتبقى صح حوالي 90% من الوقت.

**الاستنتاج:** الـ **accuracy** مش المقياس المفضّل للـ classifiers، خصوصًا مع الداتا الـ **skewed** (لما فئة أكتر بكتير من التانية).

</div>

<div dir="rtl" style="text-align:right">

### 3.2) Confusion Matrix (مصفوفة الالتباس)

طريقة أحسن بكتير للتقييم. الفكرة: **نعدّ كام مرة صور من الفئة A اتصنّفت على إنها B.**
مثلًا لو عايز تعرف كام مرة الموديل لخبط بين 5 و 3، بتبص على الصف الخامس والعمود الثالث.

عشان نحسبها محتاجين توقعات نقارنها بالقيم الحقيقية. ما ينفعش نستخدم الـ **test set** دلوقتي (هنسيبه لآخر المشروع بس). الحل: `cross_val_predict()`.

الفرق بينها وبين `cross_val_score()`: بدل ما ترجّع الـ scores، **بترجّع التوقعات** على كل fold. يعني بتاخد توقع "نضيف" لكل صورة في الـ training set (نضيف = اتعمل من موديل **ما شافش** الصورة دي في التدريب).

</div>

In [ ]:
from sklearn.model_selection import cross_val_predict

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_train_5, y_train_pred)

<div dir="rtl" style="text-align:right">

**إزاي تقرا المصفوفة؟** كل **صف** = الفئة **الحقيقية**، وكل **عمود** = الفئة **المتوقعة**.

| | متوقع: مش 5 | متوقع: 5 |
|---|---|---|
| **فعلًا مش 5** (Negative) | **TN** (True Negative): اتصنّف صح إنه مش 5 | **FP** (False Positive): مش 5 بس الموديل قال 5 ❌ |
| **فعلًا 5** (Positive) | **FN** (False Negative): كان 5 بس الموديل قال مش 5 ❌ | **TP** (True Positive): اتصنّف صح إنه 5 |

في الكتاب كانت الأرقام تقريبًا: TN = 53,272، FP = 1,307، FN = 1,077، TP = 4,344 (الأرقام عندك هتكون قريبة منها).

**الـ classifier المثالي** مفيهوش غير TP وTN، يعني الأرقام بس على القطر الرئيسي (من فوق شمال لتحت يمين).

</div>

In [ ]:
# classifier "مثالي": بيتوقع الـ labels الحقيقية بالظبط
y_train_perfect_predictions = y_train_5
confusion_matrix(y_train_5, y_train_perfect_predictions)

<div dir="rtl" style="text-align:right">

### 3.3) Precision و Recall و F1

المصفوفة بتدّيك معلومات كتير، بس أحيانًا تحتاج مقياس واحد مختصر.

**Precision (الدقة):** من اللي الموديل قال عليه "5"، كام واحد فعلًا 5؟

$$\text{precision} = \frac{TP}{TP + FP}$$

> لو عملت توقع واحد بس وطلع صح، الـ precision = 100%! بس ده classifier مش مفيد لأنه بيتجاهل كل الـ 5 التانيين. عشان كده بنستخدم Precision **مع** مقياس تاني هو Recall.

**Recall (الاسترجاع)**، وبيتسمّى كمان **Sensitivity** أو **True Positive Rate (TPR)**: من كل الـ 5 الحقيقية، كام واحد الموديل لقطه؟

$$\text{recall} = \frac{TP}{TP + FN}$$

**تشبيه بسيط:** تخيّل جهاز بيدوّر على مجرمين في الزحمة:
- **Precision** عالية = لما الجهاز يقول "ده مجرم" يبقى فعلًا مجرم (قليل من الأبرياء بيتقبض عليهم).
- **Recall** عالي = الجهاز بيلقط معظم المجرمين الحقيقيين (قليلين بيفلتوا).

</div>

In [ ]:
from sklearn.metrics import precision_score, recall_score

print("Precision:", precision_score(y_train_5, y_train_pred))   # TP / (TP + FP)
print("Recall   :", recall_score(y_train_5, y_train_pred))      # TP / (TP + FN)

<div dir="rtl" style="text-align:right">

دلوقتي الـ 5-detector ما بقاش لامع زي ما كان باين من الـ accuracy:
- لما بيقول "دي 5" يبقى صح في **حوالي 77%** بس من الحالات.
- وبيلقط **حوالي 79%** بس من الـ 5 الحقيقية.

### F1 Score

عشان نقارن classifiers بسهولة، بندمج Precision وRecall في رقم واحد هو **F1 score**، وهو **المتوسط التوافقي (Harmonic Mean)** بينهم:

$$F_1 = \frac{2}{\frac{1}{\text{precision}} + \frac{1}{\text{recall}}} = 2 \times \frac{\text{precision} \times \text{recall}}{\text{precision} + \text{recall}} = \frac{TP}{TP + \frac{FN + FP}{2}}$$

**ليه المتوسط التوافقي وليس العادي؟** لأنه بيدّي **وزن أكبر للقيم الصغيرة**. يعني ما تاخدش F1 عالي غير لو **الاتنين** (Precision وRecall) عاليين.

</div>

In [ ]:
from sklearn.metrics import f1_score

f1_score(y_train_5, y_train_pred)

<div dir="rtl" style="text-align:right">

### 3.4) الـ Precision/Recall Tradeoff

الـ F1 بيفضّل classifiers اللي الـ precision والـ recall بتوعها **متقاربين**. ده مش دايمًا اللي إنت عايزه!

**أمثلة:**
- **فيديوهات آمنة للأطفال:** تفضّل **Precision عالية** (حتى لو بيرفض فيديوهات كويسة كتير، المهم ما يعدّيش فيديو وحش أبدًا).
- **كشف السارقين في كاميرات المراقبة:** ممكن تقبل Precision 30% بس لكن **Recall 99%** (الحراس هياخدوا كام إنذار كاذب، بس تقريبًا كل السارقين هيتمسكوا).

للأسف **مش هينفع تاخد الاتنين مع بعض**: لما تزوّد الـ precision بيقل الـ recall، والعكس. ده هو الـ **Precision/Recall Tradeoff**.

#### إزاي الـ SGDClassifier بياخد قراره؟
لكل صورة بيحسب **score** من **decision function**. لو الـ score أكبر من **threshold** (عتبة) → الصورة "Positive"، لو لأ → "Negative".

- **ترفع الـ threshold** ← Precision بتزيد، Recall بيقل.
- **تنزّل الـ threshold** ← Recall بيزيد، Precision بتقل.

scikit-learn ما بتسيبكش تحدد الـ threshold مباشرة، لكن بتديك الـ scores عن طريق `decision_function()`، وإنت تقرر بنفسك.

</div>

In [ ]:
y_scores = sgd_clf.decision_function([some_digit])
print("score:", y_scores)

threshold = 0                      # ده الـ threshold الافتراضي في SGDClassifier
print("threshold = 0      ->", y_scores > threshold)

threshold = y_scores[0] + 1000     # رفعنا الـ threshold لرقم أعلى من الـ score بتاع الصورة
print("threshold =", threshold, "->", y_scores > threshold)

<div dir="rtl" style="text-align:right">

لما رفعنا الـ threshold، الصورة (اللي فعلًا 5) ما اتلقطتش، يعني **الـ recall قل**. ✔️

#### إزاي نختار الـ threshold المناسب؟
1. نجيب الـ **scores** لكل صور الـ training set باستخدام `cross_val_predict()` بس مع `method="decision_function"`.
2. نحسب Precision وRecall لكل threshold ممكن باستخدام `precision_recall_curve()`.
3. نرسم المنحنى.

</div>

In [ ]:
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3, method="decision_function")

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)

def plot_precision_recall_vs_threshold(precisions, recalls, thresholds):
    plt.plot(thresholds, precisions[:-1], "b--", label="Precision", linewidth=2)
    plt.plot(thresholds, recalls[:-1], "g-", label="Recall", linewidth=2)
    plt.xlabel("Threshold")
    plt.legend(loc="center right")
    plt.ylim([0, 1])
    plt.grid(True)

plt.figure(figsize=(8, 4))
plot_precision_recall_vs_threshold(precisions, recalls, thresholds)
plt.show()

<div dir="rtl" style="text-align:right">

> **ليه منحنى الـ Precision "مسنّن" (bumpy) أكتر من الـ Recall؟**
> لأن الـ precision **ممكن تقل أحيانًا** لما ترفع الـ threshold (مثال: من 4/5 = 80% تنزل لـ 3/4 = 75%) حتى لو في العموم بتزيد.
> أما الـ recall فـ **بتقل بس** لما ترفع الـ threshold، عشان كده منحناه ناعم.

#### طريقة تانية: رسم Precision مقابل Recall مباشرة

</div>

In [ ]:
def plot_precision_vs_recall(precisions, recalls):
    plt.plot(recalls, precisions, "b-", linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.axis([0, 1, 0, 1])
    plt.grid(True)

plt.figure(figsize=(8, 6))
plot_precision_vs_recall(precisions, recalls)
plt.show()

<div dir="rtl" style="text-align:right">

بنلاحظ إن الـ precision **بتقع بشدة** عند recall حوالي 80%. غالبًا هتختار نقطة قبل الوقعة دي (مثلًا recall حوالي 60%). بس الاختيار الفعلي بيعتمد على مشروعك.

#### مثال: عايزين Precision = 90%
الكتاب بيحدد threshold حوالي 70,000 بالنظر للرسمة. الأفضل نحسبه بالكود عشان رقمك يختلف من تشغيل لتاني:

</div>

In [ ]:
# أصغر threshold بيدّي precision >= 90%
threshold_90_precision = thresholds[np.argmax(precisions >= 0.90)]
print("threshold:", threshold_90_precision)

y_train_pred_90 = (y_scores >= threshold_90_precision)

print("Precision:", precision_score(y_train_5, y_train_pred_90))
print("Recall   :", recall_score(y_train_5, y_train_pred_90))

<div dir="rtl" style="text-align:right">

حققنا precision حوالي 90%، لكن شوف الـ recall نزل لحوالي 60% 😬

> ### 📌 قاعدة ذهبية
> لو حد قال لك **"عايزين نوصل لـ precision 99%"**، لازم تسأله: **"عند recall كام؟"**
> أي precision عالية من غير recall مقبول مالهاش لازمة.

</div>

<div dir="rtl" style="text-align:right">

### 3.5) ROC Curve

**ROC = Receiver Operating Characteristic.** أداة شائعة تانية مع الـ binary classifiers، شبيهة بمنحنى Precision/Recall، لكن بترسم:
- **المحور الصادي (Y):** TPR = Recall (Sensitivity).
- **المحور السيني (X):** **FPR** (False Positive Rate) = نسبة الـ negatives اللي اتصنفت غلط على إنها positive.

$$FPR = 1 - TNR \quad \text{حيث } TNR = \text{Specificity}$$

يعني ROC = **Sensitivity مقابل (1 − Specificity)**.

</div>

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)

def plot_roc_curve(fpr, tpr, label=None):
    plt.plot(fpr, tpr, linewidth=2, label=label)
    plt.plot([0, 1], [0, 1], "k--")      # الخط المتقطع = classifier عشوائي بحت
    plt.axis([0, 1, 0, 1])
    plt.xlabel("False Positive Rate (1 - Specificity)")
    plt.ylabel("True Positive Rate (Recall)")
    plt.grid(True)

plt.figure(figsize=(8, 6))
plot_roc_curve(fpr, tpr)
plt.show()

<div dir="rtl" style="text-align:right">

برضه فيه tradeoff: **كل ما الـ recall (TPR) يزيد، الـ false positives (FPR) بتزيد.**

- **الخط المتقطع** = classifier عشوائي تمامًا.
- الـ classifier الكويس لازم يبعد عن الخط ده قد ما يقدر (ناحية **الركن الأعلى الشمال**).

#### AUC (Area Under the Curve)
طريقة لمقارنة classifiers: نقيس **المساحة تحت المنحنى**.
- classifier **مثالي** → ROC AUC = **1**
- classifier **عشوائي** → ROC AUC = **0.5**

</div>

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_train_5, y_scores)

<div dir="rtl" style="text-align:right">

> ### 🧭 امتى أستخدم PR curve وامتى ROC؟
> **قاعدة عامة:**
> - فضّل **PR curve** لما الـ positive class **نادرة** أو لما اهتمامك بالـ **false positives** أكتر من الـ false negatives.
> - غير كده استخدم **ROC**.
>
> في مثالنا الـ ROC AUC طلع حوالي 0.96 وشكله ممتاز، لكن ده بسبب إن الـ 5 قليلة جدًا مقارنة بالباقي. أما منحنى الـ PR فبيوضّح إن لسه فيه مجال للتحسين.

### مقارنة SGD بـ Random Forest

`RandomForestClassifier` **ما عندوش** `decision_function()`، بدلها عنده **`predict_proba()`** اللي بترجّع **احتمالية** كل فئة لكل صورة (مثلًا 70% إنها 5). الـ classifiers في scikit-learn عادةً عندها واحدة من الاتنين.

عشان نرسم ROC محتاجين scores مش احتمالات، فنستخدم **احتمالية الـ positive class كـ score**.

</div>

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3,
                                    method="predict_proba", n_jobs=-1)

y_scores_forest = y_probas_forest[:, 1]   # score = احتمالية الـ positive class
fpr_forest, tpr_forest, thresholds_forest = roc_curve(y_train_5, y_scores_forest)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, "b:", linewidth=2, label="SGD")
plot_roc_curve(fpr_forest, tpr_forest, "Random Forest")
plt.legend(loc="lower right", fontsize=16)   # تعديل: الكتاب كاتب "bottom right" وده غلط
plt.show()

In [ ]:
print("ROC AUC (Random Forest):", roc_auc_score(y_train_5, y_scores_forest))

y_train_pred_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3, n_jobs=-1)
print("Precision:", precision_score(y_train_5, y_train_pred_forest))
print("Recall   :", recall_score(y_train_5, y_train_pred_forest))

<div dir="rtl" style="text-align:right">

الـ Random Forest أحسن بوضوح: منحناه أقرب للركن الأعلى الشمال والـ AUC بتاعه أعلى (حوالي 0.99)، والـ precision حوالي 98.5% والـ recall حوالي 82.8% (حسب الكتاب).

**خلاصة الجزء ده:** بقيت تعرف تدرّب binary classifiers، تختار المقياس المناسب، تقيّم بالـ cross-validation، تختار الـ precision/recall tradeoff اللي يناسبك، وتقارن موديلات بالـ ROC/AUC.

</div>

<div dir="rtl" style="text-align:right">

## 4) Multiclass Classification (تصنيف متعدد الفئات)

الـ **binary classifier** بيفرّق بين فئتين. أما الـ **multiclass** (وبيتسمّى كمان *multinomial*) فبيفرّق بين **أكتر من فئتين**.

- بعض الـ algorithms بتتعامل مع multi-class **مباشرة**: Random Forest، Naive Bayes.
- بعضها **binary بس**: Support Vector Machines، Linear classifiers.

بس فيه استراتيجيات نقدر بيها نحل multiclass باستخدام **أكتر من binary classifier**:

| الاستراتيجية | الفكرة | عدد الـ classifiers (لـ 10 أرقام) |
|---|---|---|
| **OvA** (One-versus-All) أو One-versus-the-Rest | classifier لكل رقم ("ده 0 ولا لأ؟"، "ده 1 ولا لأ؟"...). نختار الفئة اللي score بتاعها **الأعلى** | 10 |
| **OvO** (One-versus-One) | classifier لكل **زوج** من الأرقام (0 ضد 1، 0 ضد 2، ...). نختار الفئة اللي **كسبت أكتر مواجهات** | N×(N−1)/2 = **45** |

**ميزة OvO:** كل classifier بيتدرّب على جزء صغير من الداتا (الفئتين بس).
عشان كده الـ algorithms اللي بتتعب مع الداتا الكبيرة (زي **SVM**) بتفضّل OvO. أما معظم الـ algorithms التانية فبتفضّل **OvA**.

scikit-learn **بتكتشف تلقائيًا** لو استخدمت binary classifier في multiclass وبتشغّل OvA (إلا مع SVM فبتستخدم OvO).

</div>

In [ ]:
sgd_clf.fit(X_train, y_train)      # لاحظ: y_train مش y_train_5
sgd_clf.predict([some_digit])

<div dir="rtl" style="text-align:right">

بس كده! الكود ده درّب `SGDClassifier` على الأرقام من 0 لـ 9. **ورا الكواليس** scikit-learn دربت **10 binary classifiers**، جابت الـ score من كل واحد، واختارت الأعلى.

نتأكد من كده: `decision_function()` دلوقتي بترجّع **10 scores** (واحد لكل فئة).

</div>

In [ ]:
some_digit_scores = sgd_clf.decision_function([some_digit])
print(some_digit_scores)

print("أعلى score عند الفئة:", np.argmax(some_digit_scores))
print("الفئات:", sgd_clf.classes_)
print("الفئة رقم 5:", sgd_clf.classes_[5])

<div dir="rtl" style="text-align:right">

> الـ classifier بيخزّن قايمة الفئات في `classes_` **مرتّبة بالقيمة**. هنا ترتيب الفئة = قيمتها (الفئة اللي index = 5 هي 5)، لكن ده **مش بيحصل دايمًا**.

#### إجبار scikit-learn تستخدم OvO أو OvA
بنستخدم `OneVsOneClassifier` أو `OneVsRestClassifier` ونديها binary classifier:

</div>

In [ ]:
from sklearn.multiclass import OneVsOneClassifier

ovo_clf = OneVsOneClassifier(SGDClassifier(random_state=42))
ovo_clf.fit(X_train, y_train)

print(ovo_clf.predict([some_digit]))
print("عدد الـ classifiers:", len(ovo_clf.estimators_))   # 45

<div dir="rtl" style="text-align:right">

#### Random Forest مع Multiclass
الـ Random Forest بيصنّف لعدة فئات **مباشرة**، من غير OvA أو OvO. و`predict_proba()` بتديك احتمالية كل فئة:

</div>

In [ ]:
forest_clf.fit(X_train, y_train)
print(forest_clf.predict([some_digit]))

np.set_printoptions(suppress=True)
print(forest_clf.predict_proba([some_digit]))

<div dir="rtl" style="text-align:right">

الموديل متأكد إلى حد كبير (حوالي 80% إنها 5)، وشايف إن ممكن تكون 0 أو 3 باحتمالية قليلة.

### تقييم الـ Multiclass
زي العادة بنستخدم الـ cross-validation:

</div>

In [ ]:
cross_val_score(sgd_clf, X_train, y_train, cv=3, scoring="accuracy")

<div dir="rtl" style="text-align:right">

حوالي **84%** على كل الـ folds. لو الموديل بيخمّن عشوائي كان هيجيب **10%** بس، فالنتيجة مش وحشة، لكن نقدر نعمل أحسن بسهولة بـ **Feature Scaling** (زي ما اتعلمنا في الفصل التاني):

</div>

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float64))

cross_val_score(sgd_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")

<div dir="rtl" style="text-align:right">

مجرد تحجيم الـ inputs رفع الـ accuracy لأكتر من **90%**.

</div>

<div dir="rtl" style="text-align:right">

## 5) Error Analysis (تحليل الأخطاء)

في مشروع حقيقي هتتبع الـ **Machine Learning Project Checklist** (Appendix B في الكتاب): تجرّب خيارات تجهيز الداتا، موديلات كتير، تختار الأفضل منها، وتعمل fine-tuning بـ `GridSearchCV`، وهكذا.

هنا هنفترض إنك وصلت لموديل واعد وعايز تحسّنه. أحد الطرق: **تحلّل نوع الأخطاء اللي بيعملها.**

### الخطوة 1: Confusion Matrix لكل الفئات

</div>

In [ ]:
y_train_pred = cross_val_predict(sgd_clf, X_train_scaled, y_train, cv=3)
conf_mx = confusion_matrix(y_train, y_train_pred)
conf_mx

<div dir="rtl" style="text-align:right">

أرقام كتير! أسهل نعرضها كصورة بـ `matshow()`:

</div>

In [ ]:
plt.matshow(conf_mx, cmap=plt.cm.gray)
plt.show()

<div dir="rtl" style="text-align:right">

معظم الصور على **القطر الرئيسي** = اتصنفت صح. الـ 5 باينة أغمق شوية من الباقي، وده معناه إما إن فيه صور 5 أقل في الداتا، أو إن الموديل أضعف في الـ 5 (والاتنين صحيح فعلًا).

### الخطوة 2: نركّز على الأخطاء
1. نقسم كل قيمة على **عدد الصور في فئتها**، عشان نقارن **معدّلات الخطأ** مش الأعداد المطلقة (الأعداد المطلقة بتظلم الفئات الكبيرة).
2. نملا القطر بأصفار عشان نشيل الإجابات الصح ونبقي على الأخطاء بس.

</div>

In [ ]:
row_sums = conf_mx.sum(axis=1, keepdims=True)
norm_conf_mx = conf_mx / row_sums

np.fill_diagonal(norm_conf_mx, 0)
plt.matshow(norm_conf_mx, cmap=plt.cm.gray)
plt.show()

<div dir="rtl" style="text-align:right">

**إزاي نقرا الرسمة؟** (الصفوف = الفئة الحقيقية، الأعمدة = الفئة المتوقعة)

- أعمدة **8 و9** منوّرة ← كتير من الصور بتتصنّف غلط على إنها 8 أو 9.
- صفوف **8 و9** منوّرة كمان ← الـ 8 والـ 9 بيتلخبطوا مع أرقام تانية.
- صف **1** غامق ← معظم الـ 1 بتتصنّف صح.
- الأخطاء **مش متماثلة**: مثلًا فيه 5 بتتصنّف 8 أكتر من العكس.

### إيه اللي نعمله بالمعلومات دي؟
جهودنا لازم تتركز على تحسين **8 و9** وعلى لخبطة **3 و5**. أفكار:
- نجمع **داتا تدريب أكتر** للأرقام دي.
- نعمل **feature engineering**: مثلًا خوارزمية تعدّ **الحلقات المقفولة** (الـ 8 فيها اتنين، الـ 6 واحدة، الـ 5 ولا واحدة).
- **نعالج الصور مسبقًا** (باستخدام Scikit-Image أو Pillow أو OpenCV) عشان نبرّز أنماط معينة.

### تحليل أخطاء فردية (أبطأ بس مفيد)
نرسم أمثلة للـ 3 والـ 5 في الأربع حالات: صح/غلط لكل واحد منهم.

</div>

In [ ]:
cl_a, cl_b = 3, 5
X_aa = X_train[(y_train == cl_a) & (y_train_pred == cl_a)]   # 3 اتصنّفت 3 (صح)
X_ab = X_train[(y_train == cl_a) & (y_train_pred == cl_b)]   # 3 اتصنّفت 5 (غلط)
X_ba = X_train[(y_train == cl_b) & (y_train_pred == cl_a)]   # 5 اتصنّفت 3 (غلط)
X_bb = X_train[(y_train == cl_b) & (y_train_pred == cl_b)]   # 5 اتصنّفت 5 (صح)

plt.figure(figsize=(8, 8))
plt.subplot(221); plot_digits(X_aa[:25], images_per_row=5)
plt.subplot(222); plot_digits(X_ab[:25], images_per_row=5)
plt.subplot(223); plot_digits(X_ba[:25], images_per_row=5)
plt.subplot(224); plot_digits(X_bb[:25], images_per_row=5)
plt.show()

<div dir="rtl" style="text-align:right">

**الكتلتين الشمال** = صور اتصنّفت 3، **الكتلتين اليمين** = صور اتصنّفت 5. (الصحيح على القطر: فوق شمال وتحت يمين).

بعض الأخطاء سببها كتابة وحشة جدًا (حتى الإنسان هيتلخبط)، لكن معظمها **أخطاء واضحة لينا** ومش مفهوم ليه الموديل غلط فيها.

**السبب:** استخدمنا `SGDClassifier` وهو **linear model**. كل اللي بيعمله إنه بيدّي **وزن لكل pixel لكل فئة**، وبعدين بيجمع شدّة الـ pixels الموزونة عشان يطلع score لكل فئة. فبما إن الـ 3 والـ 5 بيختلفوا في **كام pixel بس** (مكان الخط الصغير اللي بيربط الجزء العلوي بالقوس السفلي) الموديل بيلخبطهم بسهولة.

**يعني الموديل حسّاس جدًا لـ shifting و rotation.** فحل ممكن لتقليل اللخبطة: **نعمل preprocessing** للصور عشان تبقى متمركزة ومش مايلة قوي، وده غالبًا هيحسّن أخطاء تانية كمان.

> 🧠 **ملاحظة:** مخّنا بيعمل pre-processing معقد جدًا قبل ما المعلومة توصل لوعينا. فإن الحاجة تبان لنا سهلة ما يعنيش إنها سهلة للموديل.

</div>

<div dir="rtl" style="text-align:right">

## 6) Multilabel Classification

لحد دلوقتي كل صورة كانت بتتصنّف لفئة **واحدة**. أحيانًا عايزين الـ classifier يطلّع **أكتر من label** للصورة الواحدة.

**مثال:** classifier بيتعرف على وشوش (Alice وBob وCharlie). لو الصورة فيها Alice وCharlie لازم يطلّع `[1, 0, 1]` ("Alice آه، Bob لأ، Charlie آه").

النظام ده بيتسمّى **multilabel classification**.

**مثال مبسّط:** لكل رقم هنعمل label مزدوج:
1. هل الرقم **كبير** (7 أو 8 أو 9)؟
2. هل الرقم **فردي**؟

هنستخدم `KNeighborsClassifier` (بيدعم multilabel، لكن مش كل الـ classifiers بتدعمه).

</div>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

y_train_large = (y_train >= 7)
y_train_odd = (y_train % 2 == 1)
y_multilabel = np.c_[y_train_large, y_train_odd]

knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train, y_multilabel)

In [ ]:
knn_clf.predict([some_digit])

<div dir="rtl" style="text-align:right">

وطلع صح: الرقم 5 **مش كبير** (`False`) و**فردي** (`True`).

### تقييم الـ Multilabel
فيه طرق كتير، والاختيار بيعتمد على مشروعك. من الطرق: نحسب **F1 score لكل label** على حدة، وبعدين ناخد **المتوسط**.

> ⏳ **تنبيه:** الخلية اللي جاية **بطيئة** (ممكن تاخد دقايق) لأن KNN بيقارن كل صورة بكل صور التدريب. لو عايز تجرّب بسرعة، استخدم عيّنة صغيرة، مثلًا `X_train[:10000]` و`y_multilabel[:10000]`.

</div>

In [ ]:
from sklearn.metrics import f1_score

# تعديل: الكتاب بيمرّر y_train هنا بالغلط، والصح y_multilabel
y_train_knn_pred = cross_val_predict(knn_clf, X_train, y_multilabel, cv=3, n_jobs=-1)
f1_score(y_multilabel, y_train_knn_pred, average="macro")

<div dir="rtl" style="text-align:right">

**`average="macro"`** معناها إن كل الـ labels ليها **نفس الأهمية**. ده مش دايمًا صح؛ لو عندك صور Alice أكتر بكتير من Bob وCharlie، ممكن تدّي كل label **وزنًا = الـ support بتاعه** (عدد العيّنات اللي عندها الـ label ده) عن طريق `average="weighted"`.

</div>

<div dir="rtl" style="text-align:right">

## 7) Multioutput Classification

آخر نوع: **multioutput-multiclass** (أو multioutput بس). هو **تعميم للـ multilabel** حيث **كل label ممكن يبقى multiclass** (أي له أكتر من قيمتين).

**مثال:** نظام **بيشيل النويز (noise) من الصور**:
- **المدخل:** صورة رقم فيها نويز.
- **المخرج:** صورة نضيفة، يعني مصفوفة شدّة pixels زي صور MNIST.

المخرج هنا **multilabel** (label لكل pixel) وكل label **قيمته من 0 لـ 255** (multiclass). يعني ده multioutput.

> الخط الفاصل بين classification و regression أحيانًا مش واضح. توقّع شدّة pixel أقرب للـ regression، وأنظمة الـ multioutput مش بس للتصنيف.

**الخطوات:** نضيف نويز عشوائي لصور MNIST بـ `np.random.randint()`، والـ targets هي الصور الأصلية.

</div>

In [ ]:
# تعديل: الكتاب بيكتب noise مرتين بنفس الاسم (غلط)، فهنا عملتهم منفصلين
noise_train = np.random.randint(0, 100, (len(X_train), 784))
noise_test = np.random.randint(0, 100, (len(X_test), 784))

X_train_mod = X_train + noise_train
X_test_mod = X_test + noise_test
y_train_mod = X_train
y_test_mod = X_test

In [ ]:
some_index = 5500

plt.subplot(121); plot_digit(X_test_mod[some_index])
plt.subplot(122); plot_digit(y_test_mod[some_index])
plt.show()

<div dir="rtl" style="text-align:right">

الشمال = الصورة النويزية (المدخل)، واليمين = الصورة النضيفة (الـ target).

> (احنا هنا بنبص على الـ test set، وده ما ينفعش في مشروع حقيقي 😄. الكتاب نفسه بيقول إنك لازم تكشّر.)

دلوقتي ندرّب الـ classifier ونخليه ينضّف الصورة:

</div>

In [ ]:
knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train_mod, y_train_mod)

clean_digit = knn_clf.predict([X_test_mod[some_index]])
plot_digit(clean_digit)
plt.show()

<div dir="rtl" style="text-align:right">

قريبة جدًا من الـ target! وبكده خلّصنا جولة التصنيف.

</div>

<div dir="rtl" style="text-align:right">

## 8) ملخص الفصل (Cheat Sheet)

### مقاييس الـ Binary Classifiers

| المقياس | الصيغة | بيجاوب على سؤال | استخدمه لما |
|---|---|---|---|
| **Accuracy** | الصح ÷ الكل | كام % من توقعاتي صح؟ | الفئات **متوازنة** (وإلا بيضلّل) |
| **Precision** | TP ÷ (TP + FP) | لما أقول "Positive" كام مرة أبقى صح؟ | غلط الـ **False Positive** مكلّف (مثال: فيديوهات الأطفال) |
| **Recall** (TPR / Sensitivity) | TP ÷ (TP + FN) | من كل الـ Positives الحقيقية كام لقطت؟ | غلط الـ **False Negative** مكلّف (مثال: كشف السرقة أو الأمراض) |
| **F1 Score** | المتوسط التوافقي بين Precision وRecall | رقم واحد يجمع الاتنين | عايز تقارن موديلات بسرعة |
| **ROC AUC** | المساحة تحت منحنى TPR مقابل FPR | قدرة الموديل على التفريق بين الفئتين | الفئات متوازنة تقريبًا |
| **PR Curve** | Precision مقابل Recall | الـ tradeoff بين الاتنين | الـ positive class **نادرة** |

### أنواع التصنيف

| النوع | عدد الـ labels | عدد القيم الممكنة لكل label | مثال |
|---|---|---|---|
| **Binary** | 1 | 2 | 5 / مش 5 |
| **Multiclass** | 1 | أكتر من 2 | الرقم من 0 لـ 9 |
| **Multilabel** | أكتر من 1 | 2 لكل label | كبير؟ فردي؟ / مين في الصورة؟ |
| **Multioutput** | أكتر من 1 | أكتر من 2 لكل label | إزالة نويز الصور |

### أهم الدروس
1. **Accuracy مضلّلة** مع الداتا الـ skewed (classifier غبي جاب 90%!).
2. استخدم **Confusion Matrix** وبعدين **Precision / Recall / F1**.
3. فيه **tradeoff** بين Precision وRecall، والـ **threshold** هو اللي بيتحكم فيه.
4. اسأل دايمًا: **"precision عند recall كام؟"**
5. **Error Analysis** بتوريك تركّز مجهودك فين.
6. **Feature scaling** ممكن ترفع الأداء بدرجة كبيرة بسهولة.
7. الـ **Random Forest** هنا أحسن من الـ SGD بفرق واضح، والموديلات الـ linear حسّاسة لـ shifting وrotation.

</div>

<div dir="rtl" style="text-align:right">

## 9) التمارين (Exercises)

الكتاب بيختم الفصل بـ 4 تمارين. هشرحلك كل واحد وهديك نقطة بداية للكود.

### تمرين 1: وصّل لـ 97% accuracy على الـ test set
**تلميح الكتاب:** الـ `KNeighborsClassifier` شغال كويس هنا. المطلوب بس تلاقي أحسن hyperparameters (جرّب grid search على `weights` و `n_neighbors`).

</div>

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [{"weights": ["uniform", "distance"], "n_neighbors": [3, 4, 5]}]

knn_clf = KNeighborsClassifier()
grid_search = GridSearchCV(knn_clf, param_grid, cv=5, verbose=3, n_jobs=-1)

# ⏳ بطيء جدًا على الداتا الكاملة (ممكن ساعات). جرّب الأول على عيّنة:
# grid_search.fit(X_train[:10000], y_train[:10000])
# بعد كده على الداتا الكاملة:
# grid_search.fit(X_train, y_train)

# print(grid_search.best_params_, grid_search.best_score_)

# from sklearn.metrics import accuracy_score
# y_pred = grid_search.predict(X_test)
# accuracy_score(y_test, y_pred)

<div dir="rtl" style="text-align:right">

### تمرين 2: Data Augmentation (توسيع الـ training set)
اكتب دالة بتزحزح صورة MNIST **بكسل واحد** في أي اتجاه (يمين، شمال، فوق، تحت). بعد كده لكل صورة في الـ training set اعمل **4 نسخ مزحزحة** وضيفهم للـ training set. دَرّب أحسن موديل عندك على الـ set الموسّع، وقيس الـ accuracy على الـ test set.

المفروض الموديل يبقى **أحسن**. التقنية دي اسمها **Data Augmentation** أو **Training Set Expansion**.

</div>

In [ ]:
from scipy.ndimage import shift   # تعديل: الكتاب بيستخدم scipy.ndimage.interpolation (اتشالت)

def shift_image(image, dx, dy):
    image = image.reshape((28, 28))
    shifted_image = shift(image, [dy, dx], cval=0, mode="constant")
    return shifted_image.reshape([-1])

# تجربة: نزحزح الصورة بكسل لتحت وبكسل للشمال
image = X_train[1000]
plt.figure(figsize=(12, 3))
plt.subplot(131); plt.title("Original");      plot_digit(image)
plt.subplot(132); plt.title("Shifted down");  plot_digit(shift_image(image, 0, 5))
plt.subplot(133); plt.title("Shifted left");  plot_digit(shift_image(image, -5, 0))
plt.show()

In [ ]:
X_train_augmented = [image for image in X_train]
y_train_augmented = [label for label in y_train]

for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
    for image, label in zip(X_train, y_train):
        X_train_augmented.append(shift_image(image, dx, dy))
        y_train_augmented.append(label)

X_train_augmented = np.array(X_train_augmented)
y_train_augmented = np.array(y_train_augmented)

# لازم نعمل shuffle تاني بعد الإضافة
shuffle_idx = np.random.permutation(len(X_train_augmented))
X_train_augmented = X_train_augmented[shuffle_idx]
y_train_augmented = y_train_augmented[shuffle_idx]

print(X_train_augmented.shape)   # (300000, 784)

# بعد كده درّب أحسن موديل (مثلًا KNN بأفضل hyperparameters من تمرين 1)
# وقيس الـ accuracy على X_test. (⏳ بطيء)

<div dir="rtl" style="text-align:right">

### تمرين 3: داتا Titanic
حلّ مشكلة **Titanic** (توقّع مين نجا). مكان البداية الممتاز هو **Kaggle**.
**خطوات مقترحة:** حمّل الداتا ← نضّف القيم الناقصة (`SimpleImputer`) ← حوّل الأعمدة النصية (`OneHotEncoder`) ← جرّب موديلات (Logistic Regression, Random Forest) ← قيّم بالـ cross-validation و Precision/Recall.

### تمرين 4: Spam Classifier (أصعب)
ابني **spam classifier**:
1. نزّل أمثلة **spam** و**ham** (إيميلات عادية) من datasets Apache SpamAssassin العامة.
2. فك الضغط واتعرّف على شكل الداتا.
3. قسّم الداتا لـ training وtest.
4. اعمل **data preparation pipeline** يحوّل كل إيميل لـ **feature vector** بيوضّح وجود/عدم وجود كل كلمة.
   - مثال: لو كل الإيميلات مكونة من 4 كلمات بس ("Hello", "how", "are", "you") فالإيميل "Hello you Hello Hello you" يتحوّل لـ `[1, 0, 0, 1]` (موجودة/مش موجودة)، أو `[3, 0, 0, 2]` لو بتعدّ التكرار.
5. أضف **hyperparameters** للـ pipeline تتحكم في: حذف headers الإيميل، تحويل لـ lowercase، حذف علامات الترقيم، استبدال الـ URLs بكلمة "URL"، استبدال الأرقام بكلمة "NUMBER"، وعمل **stemming** (قص نهايات الكلمات).
6. جرّب أكتر من classifier وحاول توصل لـ **Precision و Recall عاليين** مع بعض (وده بيربط بالـ tradeoff اللي اتعلمناه).

**نقطة بداية للتمرين 4** (تحويل النص لأرقام): `CountVectorizer` أو `TfidfVectorizer` من `sklearn.feature_extraction.text` بيعملوا بالظبط فكرة الـ word-presence/counts.

---
الحلول الرسمية للتمارين موجودة في نوتبوكات الكتاب على GitHub (مشروع `ageron/handson-ml`).

</div>